# Custom Layers and Functions

The library ships over a hundred layers, yet every one of them started life as
code somebody wrote because the layer they needed did not exist. Sooner or
later you will be in the same position: a new normalization, an unusual
residual block, an operation with no gradient. This section shows what to do.
A custom layer is a subclass of the module class from
that section with a forward method, and if you
register its state properly you inherit everything a built-in layer gets:
parameter tracking, serialization, and device movement, with no extra code. We
build up from a stateless five-liner to RMSNorm, a normalization used by many
current language models, then to layers with precomputed non-trainable state,
and finally to the case where the forward computation alone is not enough
because the gradient itself must be redefined.

In [1]:
import tensorflow as tf

## Layers without Parameters

The smallest custom layer has no state at all. `CenteredLayer` subtracts the
mean from its input. To build it, we inherit from the base layer class and
implement `call`, which Keras invokes through its own `__call__`; there is
nothing to set up, so `__init__` only calls the parent constructor.

In [2]:
class CenteredLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, X):
        return X - tf.reduce_mean(X)

Feeding data through confirms that it does what it says.

In [3]:
layer = CenteredLayer()
layer(tf.constant([1.0, 2, 3, 4, 5]))

<tf.Tensor: shape=(5,), dtype=float32, numpy=array([-2., -1.,  0.,  1.,  2.], dtype=float32)>

Nothing distinguishes this class from a built-in layer. We can place it inside
a `Sequential`, and the container neither knows nor cares that one of its
children is user code. The output mean should be zero; because we are adding
up floating-point numbers, we may see a very small nonzero value instead,
which is roundoff, not a bug.

In [4]:
net = tf.keras.Sequential([tf.keras.layers.Dense(128), CenteredLayer()])
Y = net(tf.random.uniform((4, 8)))
tf.reduce_mean(Y)

<tf.Tensor: shape=(), dtype=float32, numpy=-9.313225746154785e-10>

## Layers with Parameters: RMSNorm

A layer with something to learn must create its own parameters, and
`add_weight` is what registers them (that section). Keras
splits creation off into a `build` method that runs on the first call, once
the input shape is known, so the layer need not be told its width in advance.
We could show the mechanics by re-implementing `Dense`, but that teaches
nothing the built-in does not already do. Instead we implement *RMSNorm*
[@Zhang.Sennrich.2019], a normalization used by many current language
models.

Layer normalization standardizes each input vector: subtract the mean, divide
by the standard deviation, then apply a learned scale and shift. Zhang and
Sennrich observed that the re-centering contributes little and dropped it,
along with the shift. What remains is: divide by the root mean square,
multiply by a learned gain $\mathbf{g}$:

$$
\textrm{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} \odot \mathbf{g},
$$

where $d$ is the width of $\mathbf{x}$ and $\epsilon$ guards against division
by zero. One parameter vector, one reduction, no shift. Why dropping the mean
costs so little in practice is a question for the normalization discussion in
later chapters; here it is our stateful worked example.

In [5]:
class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def build(self, X_shape):
        self.gain = self.add_weight(name='gain', shape=[X_shape[-1]],
                                    initializer='ones')

    def call(self, X):
        rms = tf.math.rsqrt(
            tf.reduce_mean(X ** 2, axis=-1, keepdims=True) + self.eps)
        return self.gain * X * rms

Let $m_2$ denote an input row's mean square. With gain one, the output mean
square is $m_2/(m_2 + \epsilon)$. It is therefore close to one when
$m_2 \gg \epsilon$, as the badly scaled inputs below verify.

In [6]:
norm = RMSNorm()
X = 100 * tf.random.normal((4, 8))
tf.reduce_mean(norm(X) ** 2, axis=-1)

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([0.9999999, 1.0000001, 1.       , 1.       ], dtype=float32)>

### The Composability Guarantee

The reason to write RMSNorm as a module, rather than as a function with a
gain tensor floating around beside it, is what registration buys. A correctly
written custom layer is indistinguishable from a built-in one along four
axes: its parameters are tracked, it composes inside containers, its state
serializes, and it moves across devices. We check each once.

First, the gain registered itself the moment `add_weight` ran inside `build`.
Any optimizer handed `norm.trainable_variables` will find and update it.

In [7]:
norm.trainable_variables

[<Variable path=rms_norm/gain, shape=(8,), dtype=float32, value=[1. 1. 1. 1. 1. 1. 1. 1.]>]

Second, the layer drops into a `Sequential` next to built-ins.

In [8]:
net = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                           tf.keras.layers.Dense(2)])
net(tf.random.normal((4, 20))).shape

TensorShape([4, 2])

Third, its state serializes with everything else. `get_weights` returns the
model's weights, gain included, as a list of arrays, and `set_weights` loads
that list into a clone once a first call has built it; after the copy the two
models agree exactly (saving to disk works the same way; see
that section).

In [9]:
clone = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                             tf.keras.layers.Dense(2)])
X = tf.random.normal((4, 20))
clone(X)  # A first call builds the clone so its weights exist
clone.set_weights(net.get_weights())
bool(tf.reduce_all(net(X) == clone(X)))

True

Fourth, it moves, though TensorFlow settles this axis at creation rather than
on demand. There is no move-the-model call: `add_weight` places the gain on
the best available device the moment it runs (the GPU if one is visible), and
operations execute where their inputs live; a `tf.device` scope at
construction time overrides the default (that section). So there
is nothing to run here. The guarantee is the same: because the gain went
through the proper channel, placement is handled for it, custom layer or
built-in alike.

None of this took any code beyond the class definition. The guarantee comes
from the base class: subclass it, register state through the proper channels,
and containers, optimizers, checkpoints, and devices all treat your layer as
native.

### Checking against the Built-in
<!-- d2l:prose id=custom-layers-md-16 fw=pytorch, jax, tensorflow -->

RMSNorm proved useful enough that the library now ships its own
implementation. That gives us a referee. We copy a nontrivial gain into both
implementations, so that agreement cannot be an accident of default values,
and compare outputs.

In [10]:
ours, native = RMSNorm(), tf.keras.layers.RMSNormalization(epsilon=1e-6)
X = tf.random.normal((16, 8))
ours(X)  # Build both so the weights exist
native.build(X.shape)
ours.gain.assign(tf.linspace(0.5, 1.5, 8))
native.set_weights(ours.get_weights())
bool(tf.experimental.numpy.allclose(ours(X), native(X)))

True

## Precomputed State: Buffers

Some layer state is neither a parameter nor a passing activation: a causal
mask, a fixed positional table, running statistics. Such values must
persist, serialize, and move to the device together with the model — yet no
optimizer may ever touch them. Frameworks give this third kind of state its
own channel, and custom layers are where you create it.

that section introduced state that persists and travels with
the model but receives no gradient. In Keras the channel is the `trainable`
flag: `add_weight(trainable=False)` creates a variable no optimizer will
touch that still counts among the layer's weights, so it serializes through
`get_weights` and gets its device placement like any parameter. A common case
is a layer built around a precomputed table, for instance the causal mask
that keeps attention scores from looking at future positions. The mask is
fixed, so a trainable weight is wrong (the optimizer would update it) and a
plain tensor attribute is wrong too (the weights list would omit it).
BatchNorm's moving statistics are non-trainable weights by exactly this
mechanism.

One caveat before the code: the compact mask below accepts square
self-attention score matrices. Cached decoding, where query and key lengths
differ, needs an offset mask rather than this $T \times T$ slice.

In [11]:
class CausalMask(tf.keras.layers.Layer):
    def __init__(self, max_len):
        super().__init__()
        # Precompute once for the longest sequence; slice per call
        self.mask = self.add_weight(name='mask', shape=(max_len, max_len),
                                    dtype=tf.bool, trainable=False,
                                    initializer='zeros')
        self.mask.assign(
            tf.range(max_len)[None, :] > tf.range(max_len)[:, None])

    def call(self, scores):
        T = scores.shape[-1]
        return tf.where(self.mask[:T, :T], float('-inf'), scores)

The layer masks the strict upper triangle, has an empty trainable-weights
list for an optimizer to consult, and still carries the mask among its
weights.

In [12]:
mask = CausalMask(max_len=8)
print(mask(tf.zeros((3, 3))))
print(mask.trainable_weights, [w.name for w in mask.weights])

tf.Tensor(
[[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]], shape=(3, 3), dtype=float32)
[] ['mask']


## Custom Gradients

So far we customized the forward computation and let automatic
differentiation derive the backward pass. That derivation is literal: it
differentiates exactly the operations you ran. Occasionally literalness is
the problem. Quantization rounds values to a grid, and rounding is flat
almost everywhere, so its true derivative is zero. Any parameter sitting
behind a rounding operation stops learning:

In [13]:
w = tf.Variable([0.9, 1.4, 2.6])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(tf.round(w))
tape.gradient(y, w, unconnected_gradients=tf.UnconnectedGradients.ZERO)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([0., 0., 0.], dtype=float32)>

TensorFlow declares rounding non-differentiable outright, so without the
`unconnected_gradients` flag the tape reports the disconnection as `None`;
the flag asks for the zeros that calculus prescribes.

Zero gradient, exactly as calculus demands, and useless for training. The
*straight-through estimator* [@Bengio.Leonard.Courville.2013] resolves
this with a controlled lie: keep the rounding in the forward pass, but
pretend in the backward pass that it was the identity.

No automatic system can derive a lie for us, so we override the chain rule
with `@tf.custom_gradient`, a decorator under which the function returns its
backward rule alongside its output, the split the figure draws
explicitly.

![The straight-through estimator, forward versus backward. Forward keeps the true staircase round(x), close to but not the same as the identity it approximates; backward substitutes a constant surrogate gradient of 1, the identity's own derivative, for the true gradient, which is zero almost everywhere and would stop all learning.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-ste.svg)

In [14]:
@tf.custom_gradient
def round_ste(X):
    def grad(upstream):
        # Pretend forward was the identity: pass the gradient straight through
        return upstream
    return tf.round(X), grad

Two rules govern the decorator. First, the decorated function returns a
*pair*: the output and the gradient function. Because the gradient function
is a closure defined inside the forward pass, anything the backward
computation needs, the input values say, is captured for free; there is no
separate residual mechanism, and the decorated function is invoked like any
other, the decorator handling the tape bookkeeping. Second, `grad` receives
the loss gradient with respect to the output and must return one gradient per
argument of the decorated function. The genuine trap is state: if the wrapped
computation reads a `tf.Variable`, the gradient function must instead accept
a `variables` keyword argument and return the variables' gradients as a
second result; TensorFlow raises a `TypeError` naming this requirement if it
is missing.

A small example verifies that the gradient now flows. We multiply the rounded
values by known weights, so we know what gradient to expect at `w`.

In [15]:
with tf.GradientTape() as tape:
    y = tf.reduce_sum(round_ste(w) * tf.constant([1.0, 2.0, 3.0]))
tape.gradient(y, w)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([1., 2., 3.], dtype=float32)>

The downstream gradient arrives at `w` untouched, as if the rounding were the
identity, while the forward pass still computed with rounded values. Some
estimators instead zero the gradient where a saved input had saturated, while
ordinary gradient clipping bounds a large upstream gradient independently of
the quantizer. These are different approximations. Neither is a universal
default; the surrogate backward rule must match the training objective it is
meant to approximate.

A scope note to close. A custom gradient redefines the gradient of a
computation assembled from existing tensor operations; it does not create new
ones. Writing new *kernels*, code that runs on the accelerator itself, is a
separate craft that we do not cover in this book; that section
discusses how to get performance out of the operations you already have.

## Summary

A custom layer is a layer subclass: `call` defines the computation,
`add_weight` registers learnable state, and `add_weight(trainable=False)`
registers persistent state that no optimizer should touch. Registration is
what buys composability; a correctly written layer gets parameter tracking,
container compatibility, and serialization for free, with device placement
settled at creation, as we verified on RMSNorm axis by axis. When the chain
rule itself must be overridden, as in the straight-through estimator,
`@tf.custom_gradient` lets the forward function return its own backward rule.
Build custom implementations to understand them; prefer the native ones in
production.

## Exercises

1. Add an optional learned bias to `RMSNorm` (a shift applied after the
   scale, restoring part of what LayerNorm had). Verify that `get_weights` on
   a model containing it grows by the expected entry, and observe what
   `set_weights` does when handed a list saved without the bias.
1. Implement `Dropout` from scratch as a custom layer that zeroes each entry
   with probability $p$ and rescales the survivors during training, but is
   the identity during evaluation. Keras passes a `training` argument to
   `call`; how does calling an enclosing `Sequential` with `training=False`
   reach your layer?
1. Implement a clamp with a custom gradient: a `@tf.custom_gradient` function
   whose forward is `tf.clip_by_value(X, lo, hi)` and whose backward passes
   the gradient only where the input lay strictly inside the clamp range; the
   gradient closure must capture `X`. Compare its gradients with those of the
   native `tf.clip_by_value` for inputs inside, outside, and exactly on the
   boundary. Which convention does the native operation use at the boundary?
1. Write a layer that returns the leading half of the Fourier coefficients of
   its input. It has no parameters, so nothing registers. What do you still
   gain by wrapping the computation in a module instead of calling the
   transform inline wherever you need it?